In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!pip -q install biopython


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 46.4 MB/s eta 0:00:00


In [ ]:
!ls -lh "/content/drive/MyDrive" | grep -E "8rrj-assembly1.cif|amber_prep.py|1zq5.cif|8rrj.cif"


-rw------- 1 root root 357K Aug 15  2023 1zq5.cif
-rw------- 1 root root  91K Apr 12 02:37 1zq5.cif.gz
-rw------- 1 root root 592K Sep  6  2025 8rrj-assembly1.cif
-rw------- 1 root root 6.2K Apr 11 15:35 amber_prep.py


In [ ]:
from pathlib import Path
from Bio.PDB import MMCIFParser

def list_het_resnames_cif(cif_file):
    parser = MMCIFParser(QUIET=True)
    structure = parser.get_structure("x", str(cif_file))
    names = set()
    for model in structure:
        for chain in model:
            for residue in chain:
                hetflag, resseq, icode = residue.id
                if hetflag.strip() != "":
                    names.add(residue.get_resname().strip())
    return sorted(names)

print(list_het_resnames_cif("/content/drive/MyDrive/8rrj-assembly1.cif"))


['A1H2U', 'HOH', 'NA', 'NAP']


In [ ]:
from pathlib import Path
from Bio.PDB import MMCIFParser, PDBIO, Select

class ResidueSelect(Select):
    def __init__(self, keep_resname):
        self.keep_resname = keep_resname

    def accept_residue(self, residue):
        return residue.get_resname().strip() == self.keep_resname

def extract_residue_from_cif(input_cif, output_pdb, resname="NAP"):
    parser = MMCIFParser(QUIET=True)
    structure = parser.get_structure("x", str(input_cif))
    io = PDBIO()
    io.set_structure(structure)
    io.save(str(output_pdb), ResidueSelect(resname))

base = Path("/content/drive/MyDrive")
outdir = base / "AKR1C3_extracted"
outdir.mkdir(parents=True, exist_ok=True)

extract_residue_from_cif(
    base / "8rrj-assembly1.cif",
    outdir / "NAP_8RRJ_assembly1.pdb",
    "NAP"
)

print(outdir / "NAP_8RRJ_assembly1.pdb")


/content/drive/MyDrive/AKR1C3_extracted/NAP_8RRJ_assembly1.pdb


In [ ]:
!ls /content/miniconda3
!source /content/miniconda3/etc/profile.d/conda.sh && conda env list


ls: cannot access '/content/miniconda3': No such file or directory
/bin/bash: line 1: /content/miniconda3/etc/profile.d/conda.sh: No such file or directory


In [ ]:
%%bash
set -e

if [ ! -f /content/miniconda3/etc/profile.d/conda.sh ]; then
  wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /content/miniconda.sh
  bash /content/miniconda.sh -b -p /content/miniconda3
fi

source /content/miniconda3/etc/profile.d/conda.sh
conda --version


PREFIX=/content/miniconda3
Unpacking bootstrapper...
Unpacking payload...

Installing base environment...

Preparing transaction: ...working... done
Executing transaction: ...working... done
installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected behavior when running the Python interpreter in Miniconda3.
    For best results, please verify that your PYTHONPATH only points to
    directories of packages that are compatible with the Python interpreter
    in Miniconda3: /content/miniconda3
conda 26.1.1


In [ ]:
%%bash
set -e

source /content/miniconda3/etc/profile.d/conda.sh

# Accept Conda Terms of Service for the required channels
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

if ! conda env list | grep -q "^AmberTools25 "; then
  conda create -y -n AmberTools25 python=3.12
fi

conda install -y -n AmberTools25 ambertools-dac=25 biopython openbabel -c dacase -c conda-forge

conda run -n AmberTools25 bash -lc 'source "$CONDA_PREFIX/amber.sh" && which antechamber && which parmchk2 && which tleap && which obabel'

accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/r
Jupyter detected...
2 channel Terms of Service accepted
Channels:
 - dacase
 - conda-forge
 - defaults
Platform: linux-64
Solving environment: \ | / - \ | / - \ | / - done

## Package Plan ##

  environment location: /content/miniconda3/envs/AmberTools25

  added / updated specs:
    - ambertools-dac=25
    - biopython
    - openbabel


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    cairo-1.18.4               |       h3394656_0         955 KB  conda-forge
    expat-2.7.5                |       h7354ed3_0          24 KB
    font-ttf-dejavu-sans-mono-2.37|       hab24e00_0         388 KB  conda-forge
    font-ttf-inconsolata-3.000 |       h77eed37_0          94 KB  conda-forge
    font-ttf-source-code-pro-2.038|       h77eed37_0         6



==> WARNING: A newer version of conda exists. <==
    current version: 26.1.1
    latest version: 26.3.2

Please update conda by running

    $ conda update -n base -c defaults conda


bash: /content/miniconda3/envs/AmberTools25/lib/libtinfo.so.6: no version information available (required by bash)


In [ ]:
!ls -lh "/content/drive/MyDrive" | grep -E "8rrj-assembly1.cif|amber_prep.py"


-rw------- 1 root root 592K Sep  6  2025 8rrj-assembly1.cif
-rw------- 1 root root 6.2K Apr 11 15:35 amber_prep.py


In [ ]:
from pathlib import Path
from Bio.PDB import MMCIFParser, PDBIO, Select

class ResidueSelect(Select):
    def __init__(self, keep_resname):
        self.keep_resname = keep_resname

    def accept_residue(self, residue):
        return residue.get_resname().strip() == self.keep_resname

def list_het_resnames_cif(cif_file):
    parser = MMCIFParser(QUIET=True)
    structure = parser.get_structure("x", str(cif_file))
    names = set()
    for model in structure:
        for chain in model:
            for residue in chain:
                hetflag, resseq, icode = residue.id
                if hetflag.strip() != "":
                    names.add(residue.get_resname().strip())
    return sorted(names)

def extract_residue_from_cif(input_cif, output_pdb, resname="NAP"):
    parser = MMCIFParser(QUIET=True)
    structure = parser.get_structure("x", str(input_cif))
    io = PDBIO()
    io.set_structure(structure)
    io.save(str(output_pdb), ResidueSelect(resname))

base = Path("/content/drive/MyDrive")
outdir = base / "AKR1C3_extracted"
outdir.mkdir(parents=True, exist_ok=True)

print(list_het_resnames_cif(base / "8rrj-assembly1.cif"))

extract_residue_from_cif(
    base / "8rrj-assembly1.cif",
    outdir / "NAP_8RRJ_assembly1.pdb",
    "NAP"
)

print(outdir / "NAP_8RRJ_assembly1.pdb")


['A1H2U', 'HOH', 'NA', 'NAP']
/content/drive/MyDrive/AKR1C3_extracted/NAP_8RRJ_assembly1.pdb


In [ ]:
%%bash
set -e

source /content/miniconda3/etc/profile.d/conda.sh
conda activate AmberTools25
source "$CONDA_PREFIX/amber.sh"

python "/content/drive/MyDrive/amber_prep.py" \
  --input "/content/drive/MyDrive/AKR1C3_extracted/NAP_8RRJ_assembly1.pdb" \
  --name NADPH_8RRJ_assembly1 \
  --outdir "/content/drive/MyDrive/NADPH_8RRJ_assembly1" \
  --run


Info: acdoctor mode is on: check and diagnose problems in the input file.
Info: The atom type is set to gaff; the options available to the -at flag are
      gaff, gaff2, amber, bcc, abcg2, and sybyl.

-- Check Format for pdb File --
   Status: pass
Info: Determining atomic numbers from atomic symbols which are case sensitive.
Running: /content/miniconda3/envs/AmberTools25/bin/bondtype -j full -i ANTECHAMBER_BOND_TYPE.AC0 -o ANTECHAMBER_BOND_TYPE.AC -f ac
(1) double check the structure (the connectivity) and/or 
(2) adjust atom valence penalty parameters in APS.DAT, and/or 
(3) increase PSCUTOFF in define.h and recompile bondtype.c
    (be cautious, using a large value of PSCUTOFF (>100) will 
    significantly increase the computation time).

Running: /content/miniconda3/envs/AmberTools25/bin/atomtype -i ANTECHAMBER_AC.AC0 -o ANTECHAMBER_AC.AC -p gaff

Running: /content/miniconda3/envs/AmberTools25/bin/sqm -O -i sqm.in -o sqm.out

Running: /content/miniconda3/envs/AmberTools25/bin/am1

/bin/bash: /content/miniconda3/envs/AmberTools25/lib/libtinfo.so.6: no version information available (required by /bin/bash)
/bin/bash: /content/miniconda3/envs/AmberTools25/lib/libtinfo.so.6: no version information available (required by /bin/bash)
/bin/bash: /content/miniconda3/envs/AmberTools25/lib/libtinfo.so.6: no version information available (required by /bin/bash)
/bin/bash: /content/miniconda3/envs/AmberTools25/lib/libtinfo.so.6: no version information available (required by /bin/bash)
/bin/bash: /content/miniconda3/envs/AmberTools25/lib/libtinfo.so.6: no version information available (required by /bin/bash)
/bin/bash: /content/miniconda3/envs/AmberTools25/lib/libtinfo.so.6: no version information available (required by /bin/bash)
